In [1]:
import logging  #bilgi vermeyi sağlayan standart kütüphane bilgileri print ile vermektense logging içindeki farklı durumlara göre çıktı vermek daha doğru bir kullanımdır
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch #modelin sinir ağı bu kütüphane üzerinden çalışacak
import cv2
from transformers import AutoProcessor, AutoModelForMultimodalLM
#Processor image i modelin anlayabileceği sayısal verilere(çok boyutlu sayısal diziler) çevirir
#AutoModelForMultimodalLM → MiniCPM-V modelini yükler config dosyasına göre modeli yükler
#AutoProcessor            → Görsel + metni modele hazırlar

c:\Users\durak\miniconda3\envs\minicpm46_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#modelin ve processorun ortama yüklenmesi 
MODEL_ID = "openbmb/MiniCPM-V-4.6-BNB"

processor = AutoProcessor.from_pretrained(MODEL_ID) #Sadece görsel ve metni ileride nasıl hazırlayacağını bilen processor nesnesini oluşturuyor.
model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" #modelin nerede çalıştırılacağı otomatik belirlenecek (CPU veya GPU)
    )
model.eval() #eğitim değil de inference modunda çalıştırılacak. Katmanların çalışma davranışını inference'a uygun hale getirir.

W0828 14:55:34.093000 25516 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 779/779 [00:02<00:00, 389.39it/s]


MiniCPMV4_6ForConditionalGeneration(
  (model): MiniCPMV4_6Model(
    (vision_tower): MiniCPMV4_6VisionModel(
      (embeddings): MiniCPMV4_6VisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4900, 1152)
      )
      (encoder): MiniCPMV4_6VisionEncoder(
        (layers): ModuleList(
          (0-26): 27 x MiniCPMV4_6VisionEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): MiniCPMV4_6VisionAttention(
              (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affin

In [3]:
#modelin kurulumu durumu hakkında bilgilendirme
print("Model device:")
print(model.device)

print("Model class")
print(type(model))

total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_params}")

Model device:
cuda:0
Model class
<class 'transformers.models.minicpmv4_6.modeling_minicpmv4_6.MiniCPMV4_6ForConditionalGeneration'>
Total parameters: 780872944


In [4]:
#image in yüklenmesi
IMAGE_PATH = Path("test_pictures\catt.jpg")

image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise ValueError(f"Image not found at {IMAGE_PATH}")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB) #renk kanallarını değiştiriyoruz

In [5]:
#promptun düzenlenmesi
def build_prompt(user_question):
    question = f"""
Find the object requested by the user in the image.

Return only valid JSON in exactly this format:

[
    {{
        "label": "object name",
        "box": [x1, y1, x2, y2]
    }}
]

Do not write any explanation before or after the JSON.

User request:
{user_question}
"""
    return question

In [6]:
#chat modeli için inputu belirli bir konuşma yapısına getirilmesi
def build_messages(image_rgb, question):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_rgb
                },
                {
                    "type": "text",
                    "text": question
                }
            ]
        }
    ]

    return messages

In [7]:
def ask_model(processor, model, image_rgb, question):
    #mesajın oluşturulması
    messages = build_messages(
        image_rgb,
        question
    )

    #mesajın processor a verilmesi ve sonucunda inputun artık modelin anlayacağı bir formata dönüştürülmesi
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True, #girdileri tokenlara çevir
        add_generation_prompt=True, #özel tokenleri ekle
        return_dict=True, #girdileri dictionary formatında döndür
        return_tensors="pt", #çıktının tensor formatında olmasını sağla
    )

    inputs = inputs.to(model.device)
    print(inputs.keys())
    print(inputs["input_ids"])
    print(inputs["input_ids"].shape)
    
    #modelin cevap üretmesi
    with torch.inference_mode(): #inference modunda olduğumuz için gradient hesaplarını kapatıyor
        outputs = model.generate(
            **inputs,
            max_new_tokens=100, #cevabın uzunluğu 100 token ile sınırlandırılıyor
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:] #model çıktı üretirken çıkışın başında inputu da eklediği için onu kesip sadece modelin ürettiği kısmı alıyoruz
    #burada inputs["input_ids"] kısmı girdilerin dictionarysindeki girdi değerleirni alır
    #.shade[1] ile de tensorün boyutunu gösterir
    # : ile de tensör boyutu kadar olan kısımdaki tensörleri keser.

    #çıktı oluşturulurken girdinin hemen #modelin cevap üretmesi
    #arkasına yeni tokenler ekleniyor o yüzden cevabı oluşturuken promptu silmek gerekli

    # Modelin ürettiği cevabı token ID'lerinden string'e çevirme
    response = processor.batch_decode(
        generated_tokens,
        skip_special_tokens=True,  # <eos>, <pad>, <assistant> gibi özel tokenları gösterme
    )[0]  # Batch içindeki ilk cevabı al tek görsel tek soru olduğu için

    print("Modelin ham cevabı:")
    print(response)

    return response

In [ ]:
#çıktının JSON formatında olup olmadığını kontrol etme ve Python nesnesine dönüştürme
def parse_model_response(response):
        detections = json.loads(response)  # JSON stringini Python nesnesine dönüştür

        print(type(response))
        print(type(detections))
        print(detections)

        return detections

model
→ token ID tensoru
→ decode
→ string
→ json.loads()
→ Python list/dictionary

In [9]:
def scale_box_to_pixels(box, width, height):
        #box = detections[0]["box"]
        #height, width = image_bgr.shape[:2]

        x1, y1, x2, y2 = box

        x1_px = int((x1 / 1000) * width)
        y1_px = int((y1 / 1000) * height)
        x2_px = int((x2 / 1000) * width)
        y2_px = int((y2 / 1000) * height)

        print("Model koordinatları:")
        print(x1, y1, x2, y2)

        print("Piksel koordinatları:")
        print(x1_px, y1_px, x2_px, y2_px)

        return x1_px, y1_px, x2_px, y2_px 

In [10]:
def draw_bbox(image, box, label):
    height, width = image.shape[:2]

    x1_px, y1_px, x2_px, y2_px = scale_box_to_pixels(
        box,
        width,
        height
    )

    output_image = image.copy() #buradaki image a image_bgr verilecek

    cv2.rectangle(
        output_image,
        (x1_px, y1_px),
        (x2_px, y2_px),
        (0, 255, 0),
        2
    )

    cv2.putText(
        output_image,
        label,
        (x1_px, max(y1_px - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    return output_image

In [11]:
#modelin oluşturduğu resmi rgb ye çevirme ve yazdırma
def show_bbox_coordinates(box, image_bgr, label):
    result_image = draw_bbox,(
        image_bgr,
        box,
        label
    )

    result_image_rgb = cv2.cvtColor(
        result_image,
        cv2.COLOR_BGR2RGB
    )

    plt.imshow(result_image_rgb)
    plt.axis("off")
    plt.show()

In [13]:
# Uçtan uca test

user_question = "What is bbox coordinates of cat?"

# 1. Prompt oluştur
question = build_prompt(user_question)

# 2. Modelden ham cevabı al
response = ask_model(
    processor,
    model,
    image_rgb,
    question
)
print(response)
print("Modelin ham cevabı:")


# 3. JSON stringi Python nesnesine çevir
detections = parse_model_response(response)

print("\nParse edilmiş detections:")
print(detections)

# 4. İlk nesnenin bilgilerini al
first_detection = detections[0]

label = first_detection["label"]
box = first_detection["box"]

print("\nLabel:")
print(label)

print("\nModel bbox koordinatları:")
print(box)

# 5. Bbox çizilmiş resmi oluştur
result_image = draw_bbox(
    image_bgr,
    box,
    label
)

# 6. Notebook içinde göstermek için BGR -> RGB
result_image_rgb = cv2.cvtColor(
    result_image,
    cv2.COLOR_BGR2RGB
)

plt.imshow(result_image_rgb)
plt.axis("off")
plt.show()

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


KeysView({'input_ids': tensor([[248045,    846,    198, 248090,     15, 248091, 248078, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
         248056, 248056, 248056, 248056, 248079,    271,   9592,    279,   1576,
          10897,    539,    279,   1156,    303,    279,   2099,     13,    271,
           5423,   1132,   2610,   4566,    303,   6681,    411,   3443,     25,
            271,     58,    198,    262,    313,    198,    285,    328,   1448,
            763,    328,   1640,    803,    487,    198,    285,    328,   1944,
     

c:\Users\durak\miniconda3\envs\minicpm46_clean\lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Modelin ham cevabı:
[
    {
        "label": "cat",
        "box": [74 206 344 846]
]
[
    {
        "label": "cat",
        "box": [74 206 344 846]
]
Modelin ham cevabı:
[
    {
        "label": "cat",
        "box": [74 206 344 846]
]


JSONDecodeError: Expecting ',' delimiter: line 4 column 20 (char 51)